# PDA Machine Learning — DMX & Qu CAPWAP (Ton)

**Dataset:** `Dataset Laporan PDA HATTI(4).xlsx` (321 records). This notebook trains **two independent supervised Random Forest regressors**, evaluates them using project-held-out GroupKFold, analyzes engineered features and held-out permutation importance, and launches a Gradio dashboard with local SHAP explanations.

**Important:** Qu is sourced from the `QU CAPWAP Ton` column and remains in the original **Ton** units. Verify the engineering interpretation of Ton and CAPWAP Qu before publication. All four pile types, including Steel pile, are included. Actual steel pipe wall thickness is not available; no steel material sectional area or section-dependent proxy is used as a predictor.


## 1. Install packages & upload Excel

In [ ]:
%pip -q install gradio shap scikit-learn openpyxl joblib matplotlib

In [ ]:
from google.colab import files
uploaded = files.upload()
filename = next((k for k in uploaded if k.lower().endswith('.xlsx')), None)
assert filename, 'Upload the PDA Excel (.xlsx).'
print('Using:', filename)

## 2. Load, normalize and audit dataset

In [ ]:
import re, warnings, os, json, joblib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
raw = pd.read_excel(filename)
print('Rows, columns:', raw.shape)
display(raw.head(3))
print('Original columns:', list(raw.columns))

mapping = {'No':'pile_type','Dimensi ,mm':'dim_mm','Length, m':'length_m',
 'Hammer Type':'hammer','Weight Hammer, ton':'hammer_w_ton',
 'Drop Height, m':'drop_h_m','Energy Pot, Ton.m':'energy_tonm',
 'Sett,DFN,mm':'dfn_mm','Sett,DMX,mm':'dmx_mm',
 'QU CAPWAP Ton':'qu_ton',
 'Proj No,':'proj_no','Proj Area':'area'}
df = raw.rename(columns=mapping).copy()
df = df.loc[:, ~df.columns.astype(str).str.startswith('Unnamed:')]
for col in ['pile_type','hammer','proj_no','area']:
    if col in df: df[col] = df[col].astype('string').str.strip().replace({'':pd.NA})
for col in ['dim_mm','length_m','hammer_w_ton','drop_h_m','energy_tonm','dmx_mm','dfn_mm','qu_ton']:
    if col in df: df[col] = pd.to_numeric(df[col], errors='coerce')

# Qu CAPWAP is recorded in 'Ton' in the source Excel. Preserve that unit.
if 'qu_ton' not in df.columns:
    raise ValueError("Missing expected source column: QU CAPWAP Ton")
df['qu_ton'] = pd.to_numeric(df['qu_ton'],errors='coerce')
print('Qu CAPWAP (Ton):',int(df['qu_ton'].notna().sum()),'non-missing rows')
print('Qu unit kept as source label Ton; confirm whether this denotes metric ton-force.')

# Avoid merging project numbers like 535 and 535.2 without verification
# Each exact project ID stays in its own group.
df['proj_group'] = df['proj_no'].fillna('MISSING_PROJECT').astype(str)
print('Missing values:'); display(df.isna().sum().to_frame('missing'))
print('Projects:',df['proj_group'].nunique())
print('Pile types:',df['pile_type'].value_counts(dropna=False).to_dict())
print('Hammers:',df['hammer'].value_counts(dropna=False).to_dict())

## 3. Feature engineering — reproducible and auditable
Only parameters available **before** the target measurement are used. `dfn_mm` is excluded because it is another PDA output, not a prospective input. Energy is recomputed consistently from hammer weight × drop height; the supplied energy column is retained for QC only. No section-area-based proxy is used, because wall thickness for hollow/steel piles is not provided consistently. `slenderness` uses nominal dimension as a geometry ratio (not a structural slenderness check).

In [ ]:
def make_features(data):
    d = data.copy()
    d['dim_m'] = d['dim_mm'] / 1000.0
    d['energy_tonm_calc'] = d['hammer_w_ton'] * d['drop_h_m']
    # Diameter/width-based proxy, not an actual steel or hollow pile section area.
    d['slenderness'] = d['length_m'] / d['dim_m']
    return d

df = make_features(df)
feature_cols = ['length_m', 'dim_mm', 'hammer_w_ton', 'drop_h_m',
                'energy_tonm_calc', 'slenderness', 'pile_type', 'hammer']
numeric_cols = feature_cols[:6]
categorical_cols = feature_cols[6:]
# All four pile types are supported without assuming steel wall thickness.
valid = (df[numeric_cols].replace([np.inf, -np.inf], np.nan).notna().all(axis=1)
         & df[categorical_cols].notna().all(axis=1)
         & (df[['length_m', 'dim_mm', 'hammer_w_ton', 'drop_h_m']] > 0).all(axis=1)
         & df['pile_type'].isin(['Square pile', 'Spun pile', 'Bored pile', 'Steel pile']))
print('Steel pile rows in dataset:', int((df['pile_type']=='Steel pile').sum()))
print('Steel pile rows included:', int((valid & df['pile_type'].eq('Steel pile')).sum()))
print('Rows with valid inputs:',valid.sum(),'/',len(df))
if 'energy_tonm' in df:
    delta=(df['energy_tonm']-df['energy_tonm_calc']).abs()
    print('Supplied vs recomputed energy mismatches > 0.05 ton.m:',int((delta>0.05).sum()))
print('Repeated complete input rows:',int(df.loc[valid,feature_cols].duplicated().sum()))
display(df.loc[valid,feature_cols+['dmx_mm','qu_ton']].head())

## 4. Group K-Fold cross-validation (held-out projects)
Cross-validation fits preprocessing **inside each fold**, preventing scaler/encoder leakage. Each project ID is kept together. Repeated identical inputs across projects are still possible; interpret metrics accordingly. Model selection is not tuned on test folds.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def make_pipeline():
    return Pipeline([
        ('pre',ColumnTransformer([
            ('num',StandardScaler(),numeric_cols),
            ('cat',OneHotEncoder(handle_unknown='ignore'),categorical_cols)
        ])),
        ('model',RandomForestRegressor(n_estimators=400,min_samples_leaf=3,
                                       random_state=42,n_jobs=-1))
    ])

def evaluate_target(target):
    mask=valid & df[target].notna() & np.isfinite(df[target])
    sub=df.loc[mask].copy()
    print(f'{target}: {len(sub)} usable rows, {sub.proj_group.nunique()} groups')
    if len(sub)<20 or sub.proj_group.nunique()<5:
        print('Insufficient data/groups for 5-fold project-based validation.')
        return None
    X=sub[feature_cols]; y=sub[target].astype(float); groups=sub['proj_group']
    cv=GroupKFold(n_splits=5)
    results=[]; oof=pd.Series(index=sub.index,dtype=float)
    for i,(tr,te) in enumerate(cv.split(X,y,groups),1):
        m=make_pipeline(); m.fit(X.iloc[tr],y.iloc[tr])
        p=m.predict(X.iloc[te]); oof.iloc[te]=p
        results.append({'Fold':i,'Train':len(tr),'Test':len(te),
          'Test projects':groups.iloc[te].nunique(),
          'MAE':mean_absolute_error(y.iloc[te],p),
          'RMSE':np.sqrt(mean_squared_error(y.iloc[te],p)),
          'R2':r2_score(y.iloc[te],p)})
    table=pd.DataFrame(results); display(table.round(3))
    print('Overall out-of-fold:',{'MAE':round(mean_absolute_error(y,oof),3),
      'RMSE':round(np.sqrt(mean_squared_error(y,oof)),3),
      'R2':round(r2_score(y,oof),3)})
    fig,axs=plt.subplots(1,2,figsize=(11,4))
    axs[0].scatter(y,oof,alpha=.65); lim=[min(y.min(),oof.min()),max(y.max(),oof.max())]
    axs[0].plot(lim,lim,'--',color='gray');axs[0].set(xlabel='Actual',ylabel='OOF predicted',title=f'{target}: held-out projects')
    axs[1].hist(oof-y,bins=20);axs[1].set(xlabel='Predicted - actual',ylabel='Count',title='OOF residuals')
    plt.tight_layout();plt.show()
    final=make_pipeline().fit(X,y)
    return {'model':final,'metrics':table,'oof':pd.DataFrame({'actual':y,'predicted':oof,'project':groups}),
            'training':sub,'target':target}

models={}
models['dmx_mm']=evaluate_target('dmx_mm')
if df['qu_ton'].notna().sum():
    print('Qu CAPWAP target uses source units: Ton (unit interpretation requires verification).')
    models['qu_ton']=evaluate_target('qu_ton')
else:
    print('Qu training SKIPPED — no numeric Qu records.')
models={k:v for k,v in models.items() if v is not None}
print('TRAINED MODELS:', list(models))
print('QU NUMERIC ROWS:', int(df['qu_ton'].notna().sum()))
print('QU VALID TRAINING ROWS:', int((valid & df['qu_ton'].notna() & np.isfinite(df['qu_ton'])).sum()))

## 5. Feature engineering review & global importance

- `energy_tonm_calc = hammer_w_ton × drop_h_m`; compare with recorded `Energy Pot, Ton.m` (rounding differences possible).
- `slenderness = length_m / (dim_mm/1000)`.
- No area-based derived predictor is used. Steel pile is included without assuming its wall thickness. `slenderness` uses nominal dimension only, so interpret it as a geometry ratio.
- Engineered features are correlated with their parent variables. Permutation importance can share/reallocate importance across correlated features; rankings are **predictive associations, not causal effects**.
- For both targets, importance is calculated on held-out projects using the same GroupKFold scheme, avoiding in-sample importance inflation.

In [ ]:
from sklearn.inspection import permutation_importance

importance_results = {}
for target, item in models.items():
    sub = item['training']
    X_i, y_i = sub[feature_cols], sub[target].astype(float)
    groups_i = sub['proj_group']
    rows = []
    for fold, (tr, te) in enumerate(GroupKFold(n_splits=5).split(X_i,y_i,groups_i),1):
        m = make_pipeline().fit(X_i.iloc[tr],y_i.iloc[tr])
        perm = permutation_importance(m, X_i.iloc[te], y_i.iloc[te],
                                      n_repeats=5, random_state=42,
                                      scoring='neg_mean_absolute_error', n_jobs=-1)
        for feat, value in zip(feature_cols, perm.importances_mean):
            rows.append({'fold':fold,'feature':feat,'MAE increase when shuffled':value})
    imp = (pd.DataFrame(rows).groupby('feature')['MAE increase when shuffled']
           .agg(['mean','std']).sort_values('mean',ascending=False))
    importance_results[target]=imp
    print('\nHeld-out permutation importance:',target)
    display(imp.round(4))
    imp['mean'].sort_values().plot.barh(figsize=(8,4),title=f'{target}: permutation importance')
    plt.xlabel('Increase in MAE (target units)');plt.tight_layout();plt.show()


## 5. Input coverage and SHAP explanation
Coverage is **not a probability/confidence score**. SHAP explains the model’s prediction, not physical causality. Correlated engineered features share attribution; do not treat individual SHAP rankings as independent causal effects.

In [ ]:
import shap
explainers={}
for target,item in models.items():
    explainers[target]=shap.TreeExplainer(item['model'].named_steps['model'])

def analyze_one(target,pile_type,length_m,dim_mm,hammer,hammer_w_ton,drop_h_m):
    if target not in models: raise ValueError(f'{target} model is not trained.')
    vals=[length_m,dim_mm,hammer_w_ton,drop_h_m]
    if any(not np.isfinite(float(v)) or float(v)<=0 for v in vals):
        raise ValueError('All numeric inputs must be positive finite numbers.')
    if pile_type not in ['Square pile','Spun pile','Bored pile','Steel pile']:
        raise ValueError('Pile type not supported by the trained model.')
    row=make_features(pd.DataFrame([{'pile_type':pile_type,'length_m':float(length_m),
        'dim_mm':float(dim_mm),'hammer':hammer,'hammer_w_ton':float(hammer_w_ton),
        'drop_h_m':float(drop_h_m)}]))[feature_cols]
    item=models[target];pipe=item['model'];pred=float(pipe.predict(row)[0])
    train=item['training']; notes=[]
    for c in numeric_cols:
        low,high=train[c].min(),train[c].max();v=row[c].iloc[0]
        if not low<=v<=high: notes.append(f'{c}: {v:.3f} outside training range [{low:.3f}, {high:.3f}]')
    for c in categorical_cols:
        if row[c].iloc[0] not in set(train[c].dropna()): notes.append(f'{c}: unseen category')
    pre=pipe.named_steps['pre'];z=pre.transform(row)
    if hasattr(z,'toarray'):z=z.toarray()
    z=np.asarray(z)
    sv=explainers[target](z)
    explanation=shap.Explanation(values=sv.values[0],
       base_values=float(np.asarray(sv.base_values).reshape(-1)[0]),
       data=z[0],feature_names=pre.get_feature_names_out())
    plt.figure(figsize=(11,7))
    shap.plots.waterfall(explanation,max_display=12,show=False)
    fig=plt.gcf();fig.tight_layout()
    table=pd.DataFrame({'Feature':explanation.feature_names,
      'SHAP (target units)':explanation.values})
    table['Absolute']=table['SHAP (target units)'].abs()
    table=table.sort_values('Absolute',ascending=False).round(4)
    return pred,fig,table,notes

print('Models available:',list(models))

## 6. Interactive Gradio dashboard — DMX + optional Qu

In [ ]:
import gradio as gr
pile_choices=sorted(set(df.loc[valid,'pile_type'].dropna()))
hammer_choices=sorted(set(df.loc[valid,'hammer'].dropna()))
target_choices=[('DMX (mm)','dmx_mm')]+([('Qu CAPWAP (Ton)','qu_ton')] if 'qu_ton' in models else [])

def run_dashboard(target,pile,length,dimension,hammer,weight,drop):
    try:
        prediction,fig,table,warnings_=analyze_one(target,pile,length,dimension,hammer,weight,drop)
        unit='mm' if target=='dmx_mm' else 'Ton'
        status=('⚠️ Outside training coverage:\n'+'\n'.join(warnings_)) if warnings_ else 'Within marginal training ranges (not a confidence guarantee).'
        return f'{prediction:.3f} {unit}',fig,table,status
    except Exception as e:
        return 'Prediction unavailable',None,pd.DataFrame(),f'Error: {e}'

with gr.Blocks(title='PDA Prediction Dashboard') as demo:
    gr.Markdown('# PDA Prediction Dashboard')
    gr.Markdown('### Random Forest | DMX & Qu prediction with SHAP explanations')
    if 'qu_ton' not in models:
        gr.Markdown('⚠️ **Qu model not available:** Inspect the target validation/training output in Section 4.')
    with gr.Row():
        target=gr.Dropdown(choices=target_choices,value='dmx_mm',label='Prediction target')
        pile=gr.Dropdown(choices=pile_choices,value='Square pile' if 'Square pile' in pile_choices else pile_choices[0],label='Pile type')
        length=gr.Number(value=9,label='Pile length (m)')
        dim=gr.Number(value=250,label='Pile dimension (mm)')
    with gr.Row():
        hammer=gr.Dropdown(choices=hammer_choices,value='Drop Hammer' if 'Drop Hammer' in hammer_choices else hammer_choices[0],label='Hammer type')
        weight=gr.Number(value=2,label='Hammer weight (ton)')
        drop=gr.Number(value=1.5,label='Drop height (m)')
    btn=gr.Button('PREDICT & EXPLAIN',variant='primary')
    result=gr.Textbox(label='Prediction')
    coverage=gr.Textbox(label='Input coverage / limitations',lines=3)
    plot=gr.Plot(label='SHAP waterfall')
    table=gr.Dataframe(label='Local SHAP contributions')
    btn.click(run_dashboard,[target,pile,length,dim,hammer,weight,drop],
              [result,plot,table,coverage])
    gr.Markdown('**Disclaimer:** Model output is exploratory; coverage is not confidence; SHAP is not causal evidence.')

print('AVAILABLE MODELS:', sorted(models))
print('DROPDOWN PILE TYPES:', pile_choices)
demo.launch(share=True,debug=False)

## 7. Save models and metadata for deployment
Stop the Gradio cell above before continuing if it blocks execution. The exported package is a Python pickle/joblib: **load only files you created and trust**. Deployment requires compatible library versions.

In [ ]:
import sklearn, gradio, shap
os.makedirs('pda_export',exist_ok=True)
for target,item in models.items():
    joblib.dump(item['model'],f'pda_export/{target}_model.joblib')
metadata={'features':feature_cols,'numeric':numeric_cols,'categorical':categorical_cols,
 'targets':list(models),'pile_choices':pile_choices,'hammer_choices':hammer_choices,
 'sklearn_version':sklearn.__version__,'gradio_version':gradio.__version__,
 'shap_version':shap.__version__,'notes':'Qu CAPWAP preserved in source Ton units; confirm ton-force interpretation; no area-based engineering proxy is used.'}
with open('pda_export/metadata.json','w') as f:json.dump(metadata,f,indent=2)
print(metadata)
import shutil
shutil.make_archive('pda_export','zip','pda_export')
from google.colab import files
files.download('pda_export.zip')

## Hugging Face Spaces — deployment after validating both targets

1. Finish running this notebook and verify both DMX and Qu predictions, cross-validation results and SHAP explanations.
2. Run the export cell to download `pda_export.zip` (models + metadata).
3. Create a free account at https://huggingface.co and open **Spaces → Create new Space**.
4. Choose **Gradio** as the SDK. Choose visibility carefully: a public Space exposes the app and any files committed to the repository. Do not upload the source Excel or confidential CAPWAP reports.
5. The Space needs an `app.py` implementing the same `make_features` and `analyze_one` logic, a `requirements.txt` with compatible pinned package versions, and exported `.joblib` models. This notebook **does not yet generate a complete deployment-ready `app.py`**.
6. Test a fixed input against Colab for both DMX and Qu, and compare predictions and SHAP plots before sharing.
7. Free Spaces may sleep when idle and take time to wake up. The Gradio share URL created by Colab is temporary and requires the Colab runtime to remain active.

**Next deployment step:** Generate a self-contained `app.py` and requirements from the validated notebook; do not deploy until input preprocessing and unit conventions have been checked.
